# Tree-Based Methods

Tree-based methods are among the most practically useful tools in supervised machine learning. Unlike the linear models covered in the previous notebook, they make no parametric assumptions about the functional form of $f\colon \mathcal{X} \to \mathcal{Y}$, and they handle heterogeneous feature types, nonlinear interactions, and missing data with minimal preprocessing. A single decision tree is a weak learner — high variance, interpretable, and easily overfit. The real power comes from ensembles: **random forests** reduce variance through bagging and decorrelation, while **gradient boosting** reduces bias by iteratively fitting residuals. Both dominate tabular benchmarks when neural networks are not the right tool.

## Decision Trees

A **decision tree** partitions the feature space $\mathcal{X} = \mathbb{R}^d$ into a set of axis-aligned rectangular regions $\{R_m\}_{m=1}^M$ and assigns a constant prediction to each region. For classification the constant is the majority class; for regression it is the mean target value. The model is built by **recursive binary splitting**: starting from the root, at each node we select a feature $j$ and a threshold $t$ that minimize an impurity criterion over the resulting child nodes. This continues until some stopping criterion is met.

**Impurity criteria.** For classification with $K$ classes, the two dominant criteria are **Gini impurity** and **information gain**. Let $\hat{p}_{mk}$ be the proportion of class-$k$ samples in region $R_m.$ The Gini impurity of a node is

$$G(R_m) = \sum_{k=1}^{K} \hat{p}_{mk}(1 - \hat{p}_{mk}) = 1 - \sum_{k=1}^K \hat{p}_{mk}^2.$$

It measures expected misclassification if we randomly assigned labels according to $\hat{p}_{mk}$: a pure node with $\hat{p}_{mk} = 1$ for some $k$ has $G = 0$. **Information gain** uses the entropy of the node distribution instead:

$$H(R_m) = -\sum_{k=1}^{K} \hat{p}_{mk} \log \hat{p}_{mk}.$$

Given a candidate split that divides $R_m$ into left child $R_L$ and right child $R_R$ with $n_L$ and $n_R$ samples respectively, the **weighted impurity reduction** is

$$\Delta = I(R_m) - \frac{n_L}{n_L + n_R} I(R_L) - \frac{n_R}{n_L + n_R} I(R_R)$$

where $I$ is either $G$ or $H.$ The greedy algorithm exhaustively searches over all $j$ and $t$ and selects the split maximizing $\Delta.$ In scikit-learn, `criterion='gini'` is the default; both criteria produce qualitatively similar trees in practice.

**Trees as piecewise-constant functions.** The resulting hypothesis $h\colon \mathbb{R}^d \to \mathbb{R}$ takes the form

$$h(\mathbf{x}) = \sum_{m=1}^{M} c_m \cdot \mathbf{1}[\mathbf{x} \in R_m]$$

where $c_m$ is the leaf constant for region $R_m.$ This is exactly a piecewise-constant function whose discontinuities are aligned with the coordinate axes — it cannot express a decision boundary diagonal to the feature axes without many splits.

**Pruning.** Unrestricted recursive splitting memorizes training data, producing zero training error on most datasets. Two strategies control this. **Pre-pruning** halts growth early via hyperparameters such as `max_depth` (maximum number of levels from root to leaf) and `min_samples_leaf` (minimum number of training samples a leaf must contain). **Post-pruning** — implemented in scikit-learn via the `ccp_alpha` hyperparameter — grows the full tree first, then prunes back branches whose removal costs less than `ccp_alpha` per node removed, where cost is measured in terms of the training impurity increase. Cross-validating `ccp_alpha` is often more principled than manual depth tuning.

**Feature splits at each internal node.** scikit-learn's `DecisionTreeClassifier` uses randomized splits — it considers `max_features` features at each node and selects the best split among them. When `max_features=None` all features are considered; this is the standard single-tree setting. Restricting `max_features` is the key trick used later in random forests to decorrelate trees.

Fitting a decision tree on the breast cancer dataset and visualizing how accuracy varies with depth:

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats("svg")

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

depths = range(1, 16)
train_accs, test_accs = [], []
for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, clf.predict(X_train)))
    test_accs.append(accuracy_score(y_test, clf.predict(X_test)))

The train and test accuracy curves illustrate overfitting as depth increases:

In [ ]:
#| code-fold: true
plt.figure(figsize=(6, 4))
plt.plot(depths, train_accs, color="C0", linewidth=2, label="train")
plt.plot(depths, test_accs, color="C1", linewidth=2, label="test")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.grid(linestyle="dotted", alpha=0.6)
plt.legend();

**Figure.** Training accuracy reaches 1.0 at depth 6 while test accuracy peaks around depth 4–5 and then declines — the classic signature of a high-variance model memorizing the training set.

We can also post-prune by cross-validating `ccp_alpha`. scikit-learn exposes the pruning path through `.cost_complexity_pruning_path`:

In [ ]:
from sklearn.model_selection import cross_val_score

full_tree = DecisionTreeClassifier(random_state=42)
path = full_tree.cost_complexity_pruning_path(X_train, y_train)
alphas = path.ccp_alphas[:-1]  # exclude the trivially-pruned root

cv_scores = []
for alpha in alphas:
    clf = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    scores = cross_val_score(clf, X_train, y_train, cv=5)
    cv_scores.append(scores.mean())

best_alpha = alphas[np.argmax(cv_scores)]
pruned_tree = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=42)
pruned_tree.fit(X_train, y_train)
print(f"best ccp_alpha: {best_alpha:.5f}")
print(f"pruned tree depth: {pruned_tree.get_depth()}")
print(f"test accuracy:  {accuracy_score(y_test, pruned_tree.predict(X_test)):.4f}")

## Random Forests

A single decision tree suffers from high variance: small changes in the training set can produce very different trees. **Bagging** (bootstrap aggregating) reduces variance by training $B$ independent models on bootstrap resamples of the training data and averaging their predictions. If $\hat{f}_1, \ldots, \hat{f}_B$ are i.i.d. estimators each with variance $\sigma^2$, their mean has variance $\sigma^2 / B.$ This reduction comes with no cost to bias, since each tree is trained on a dataset of the same size.

**Decorrelating trees via random feature subsampling.** In practice, bagged trees are not i.i.d.: if one feature dominates the data, nearly every tree will split on it at the root, making the $B$ trees highly correlated. For correlated estimators with pairwise correlation $\rho$, the ensemble variance is

$$\text{Var}\!\left(\bar{f}\right) = \rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2.$$

As $B \to \infty$ the second term vanishes but the first does not — the irreducible floor is $\rho \sigma^2.$ **Random forests** reduce $\rho$ by restricting each split to a random subset of $m \ll d$ features. The standard default is $m = \lfloor\sqrt{d}\rfloor$ for classification and $m = \lfloor d/3 \rfloor$ for regression, controlled by `max_features` in scikit-learn.

**Out-of-bag error.** Each bootstrap resample includes roughly $1 - e^{-1} \approx 63.2\%$ of the training samples; the remaining $\approx 36.8\%$ are the **out-of-bag** (OOB) samples for that tree. We can predict each training sample by averaging only the trees for which it was OOB. This gives a nearly unbiased estimate of generalization error at no extra computational cost — effectively a free cross-validation.

**Key hyperparameters.** The most important knobs to tune are (1) `n_estimators`: more trees always helps, but with diminishing returns and increasing cost — 100–500 is a common range; (2) `max_features`: the randomization lever for decorrelation, try `'sqrt'`, `'log2'`, or a float fraction; (3) `max_depth` and `min_samples_leaf`: depth control to prevent individual trees from overfitting; (4) `bootstrap`: setting `False` removes the resampling step and produces a simple feature-randomized ensemble.

Fitting a random forest and reading off the OOB score:

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_features="sqrt",
    oob_score=True,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

print(f"OOB accuracy:   {rf.oob_score_:.4f}")
print(f"test accuracy:  {accuracy_score(y_test, rf.predict(X_test)):.4f}")

The OOB estimate tracks the held-out test accuracy closely without requiring a separate validation split. We can also inspect how the ensemble error evolves with the number of trees:

In [ ]:
#| code-fold: true
from sklearn.ensemble import RandomForestClassifier

n_estimators_range = range(1, 301, 5)
oob_errors, test_errors = [], []

for n in n_estimators_range:
    clf = RandomForestClassifier(
        n_estimators=n, max_features="sqrt", oob_score=True,
        random_state=42, n_jobs=-1,
    )
    clf.fit(X_train, y_train)
    oob_errors.append(1 - clf.oob_score_)
    test_errors.append(1 - accuracy_score(y_test, clf.predict(X_test)))

plt.figure(figsize=(6, 4))
plt.plot(n_estimators_range, oob_errors, color="C0", linewidth=2, label="OOB error")
plt.plot(n_estimators_range, test_errors, color="C1", linewidth=2, label="test error")
plt.xlabel("n_estimators")
plt.ylabel("error rate")
plt.grid(linestyle="dotted", alpha=0.6)
plt.legend();

**Figure.** Both OOB and test error fall quickly with the first few dozen trees and plateau around $n = 100$. Adding more trees never hurts — it just costs compute.

## Gradient Boosting: XGBoost and LightGBM

While bagging reduces variance by averaging independent models, **boosting** reduces bias by building models sequentially, each one correcting the errors of the ensemble so far. The core idea is simple: fit the current model to the **residuals** of the previous ensemble. For regression with squared loss, if $F_{t-1}(\mathbf{x})$ is the ensemble after $t-1$ steps, the $t$-th tree is fit to $r_i = y_i - F_{t-1}(\mathbf{x}_i)$, and the updated ensemble is $F_t(\mathbf{x}) = F_{t-1}(\mathbf{x}) + \eta \, h_t(\mathbf{x})$ for a learning rate $\eta > 0$.

**Gradient boosting as functional gradient descent.** Friedman's insight is that fitting residuals is equivalent to gradient descent in the space of functions. For any differentiable loss $\ell$, the negative **functional gradient** with respect to $F(\mathbf{x}_i)$ is

$$r_i = -\left[\frac{\partial \ell(y_i, F(\mathbf{x}_i))}{\partial F(\mathbf{x}_i)}\right]_{F = F_{t-1}}.$$

For squared loss $\ell = \frac{1}{2}(y - F)^2$ this gives $r_i = y_i - F_{t-1}(\mathbf{x}_i)$ — the usual residual. For log-loss, $r_i = y_i - p_i$ where $p_i = \sigma(F_{t-1}(\mathbf{x}_i))$ is the current predicted probability. Boosting thus generalizes to any loss automatically by changing what quantity each tree is asked to approximate.

**XGBoost: second-order Taylor expansion.** XGBoost goes further by using a second-order Taylor expansion of the loss around $F_{t-1}$:

$$\mathcal{L}^{(t)} \approx \sum_{i=1}^N \left[ g_i h_t(\mathbf{x}_i) + \frac{1}{2} h_i \, h_t(\mathbf{x}_i)^2 \right] + \Omega(h_t)$$

where $g_i = \partial_{\hat{y}} \ell(y_i, \hat{y})|_{\hat{y}=F_{t-1}(\mathbf{x}_i)}$ and $h_i = \partial^2_{\hat{y}} \ell(y_i, \hat{y})|_{\hat{y}=F_{t-1}(\mathbf{x}_i)}$ are the first and second derivatives (gradients and Hessians), and $\Omega(h_t) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^T w_j^2$ is a regularization term penalizing the number of leaves $T$ and the squared leaf weights $w_j.$ Minimizing this objective analytically yields the **optimal leaf weight** for leaf $j$:

$$w_j^* = -\frac{\sum_{i \in R_j} g_i}{\sum_{i \in R_j} h_i + \lambda}$$

and the corresponding **gain** for evaluating a split is

$$\text{Gain} = \frac{1}{2}\left[ \frac{\left(\sum_{i \in R_L} g_i\right)^2}{\sum_{i \in R_L} h_i + \lambda} + \frac{\left(\sum_{i \in R_R} g_i\right)^2}{\sum_{i \in R_R} h_i + \lambda} - \frac{\left(\sum_{i \in R_j} g_i\right)^2}{\sum_{i \in R_j} h_i + \lambda} \right] - \gamma.$$

The second-order information makes XGBoost more data-efficient per tree and the closed-form leaf weights make regularization precise.

**LightGBM: histogram-based splits and leaf-wise growth.** LightGBM introduces two speed improvements over XGBoost. First, instead of sorting feature values to find split thresholds, it **bins** continuous features into at most $B$ bins (default 255) and finds the optimal bin boundary — this reduces split-finding from $O(Nd)$ to $O(Bd)$ per level. Second, LightGBM grows trees **leaf-wise** (best-first): at each step it expands the leaf with the highest gain regardless of depth, rather than level-by-level. This produces deeper, more asymmetric trees that often converge faster, but requires `num_leaves` to be controlled carefully to avoid overfitting.

**Key hyperparameters.** For both XGBoost and LightGBM the most critical knobs are: (1) `n_estimators`: number of boosting rounds — use early stopping rather than tuning this directly; (2) `learning_rate` ($\eta$): smaller values require more rounds but generalize better, typically in $[0.01, 0.3]$; (3) `max_depth` (XGBoost) / `num_leaves` (LightGBM): tree complexity; (4) `subsample` and `colsample_bytree`: row and column subsampling fractions, which introduce stochasticity and reduce overfitting; (5) `reg_lambda` ($\lambda$) and `reg_alpha`: L2 and L1 leaf regularization.

**Data.** We use the California housing dataset — a regression task with $N = 20{,}640$ samples and $d = 8$ features — to demonstrate XGBoost training with early stopping:

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
X_h, y_h = housing.data, housing.target
X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42
)
X_h_train, X_h_val, y_h_train, y_h_val = train_test_split(
    X_h_train, y_h_train, test_size=0.1, random_state=42
)

print(f"train: {X_h_train.shape}  val: {X_h_val.shape}  test: {X_h_test.shape}")
print(f"features: {housing.feature_names}")

**Training.** We pass `eval_set` to XGBoost so it reports validation RMSE at each round and halts if the metric does not improve for `early_stopping_rounds` consecutive rounds:

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    eval_metric="rmse",
    early_stopping_rounds=50,
    verbosity=0,
)
xgb_model.fit(
    X_h_train, y_h_train,
    eval_set=[(X_h_val, y_h_val)],
    verbose=100,
)

Early stopping selects the number of rounds for us. We can inspect the final test RMSE and compare against LightGBM:

In [ ]:
import lightgbm as lgb
from sklearn.metrics import root_mean_squared_error

lgb_model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,
)
lgb_model.fit(
    X_h_train, y_h_train,
    eval_set=[(X_h_val, y_h_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)],
)

xgb_rmse = root_mean_squared_error(y_h_test, xgb_model.predict(X_h_test))
lgb_rmse = root_mean_squared_error(y_h_test, lgb_model.predict(X_h_test))
print(f"XGBoost test RMSE:  {xgb_rmse:.4f}  (best round: {xgb_model.best_iteration})")
print(f"LightGBM test RMSE: {lgb_rmse:.4f}  (best round: {lgb_model.best_iteration_})")

We can visualize how validation RMSE evolves over boosting rounds for both models:

In [ ]:
#| code-fold: true
xgb_results = xgb_model.evals_result()["validation_0"]["rmse"]
lgb_results = lgb_model.evals_result_["valid_0"]["rmse"]

plt.figure(figsize=(7, 4))
plt.plot(xgb_results, color="C0", linewidth=1.5, label="XGBoost val RMSE")
plt.plot(lgb_results, color="C1", linewidth=1.5, label="LightGBM val RMSE")
plt.axvline(xgb_model.best_iteration, color="C0", linestyle="dashed", lw=0.9, alpha=0.7)
plt.axvline(lgb_model.best_iteration_, color="C1", linestyle="dashed", lw=0.9, alpha=0.7)
plt.xlabel("boosting round")
plt.ylabel("RMSE")
plt.grid(linestyle="dotted", alpha=0.6)
plt.legend();

**Figure.** Both models converge rapidly in the first 200 rounds and level off. The dashed verticals mark where early stopping triggered. LightGBM typically converges faster per round due to histogram-based splits and leaf-wise growth.

The three-panel figure below compares decision boundaries on synthetic 2D data across a single tree, random forest, and XGBoost:

In [ ]:
#| label: fig-decision-boundary
#| fig-cap: "Decision boundaries learned by a single decision tree, random forest, and XGBoost on a synthetic 2D classification dataset. Boosting produces smoother, more accurate boundaries."
#| code-fold: true

from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

X_2d, y_2d = make_moons(n_samples=400, noise=0.25, random_state=42)
X_2d_train, X_2d_test, y_2d_train, y_2d_test = train_test_split(
    X_2d, y_2d, test_size=0.25, random_state=42
)

models = [
    ("Decision Tree\n(max_depth=5)",
     DecisionTreeClassifier(max_depth=5, random_state=42)),
    ("Random Forest\n(n=200)",
     RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ("XGBoost",
     xgb.XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
                        use_label_encoder=False, eval_metric="logloss",
                        random_state=42, verbosity=0)),
]

for _, m in models:
    m.fit(X_2d_train, y_2d_train)

xx, yy = np.meshgrid(
    np.linspace(X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5, 300),
    np.linspace(X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5, 300),
)
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, m) in zip(axes, models):
    Z = m.predict(grid).reshape(xx.shape)
    acc = accuracy_score(y_2d_test, m.predict(X_2d_test))
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="RdBu")
    ax.scatter(X_2d_train[:, 0], X_2d_train[:, 1],
               c=y_2d_train, cmap="RdBu", edgecolors="k", s=20, lw=0.4)
    ax.set_title(f"{name}\ntest acc = {acc:.2f}")
    ax.set_xticks([]); ax.set_yticks([])

fig.tight_layout();

## Feature Importance

Once a model is trained, a natural question is: which features drove its predictions? Tree ensembles support two broad approaches: **impurity-based importance** and **permutation importance**.

**Impurity-based (mean decrease in impurity).** Scikit-learn's `.feature_importances_` attribute reports, for each feature $j$, the total weighted impurity decrease accumulated across all splits that used feature $j$, normalized so the importances sum to 1:

$$\text{Imp}(j) = \frac{1}{|\text{Trees}|} \sum_{t} \sum_{\text{node} \in t, \; j_n = j} \frac{n_n}{N} \cdot \Delta I_n$$

where $n_n$ is the number of samples at node $n$ and $\Delta I_n$ is the impurity reduction from that split. This is extremely fast to compute — it is a byproduct of training — but it is [well-known to be biased toward high-cardinality features]{.mark}: a feature with many distinct values creates more candidate split thresholds and is therefore more likely to be selected by chance, inflating its apparent importance.

:::{.callout-caution}
Impurity-based importance is unreliable when features differ substantially in cardinality or when features are correlated. A random numeric feature with many unique values can outscore a genuinely important low-cardinality feature. Use permutation importance or SHAP for trustworthy rankings.

:::

**Permutation importance.** For a fitted model $f$ and a metric $s$ (e.g. accuracy or $R^2$), the permutation importance of feature $j$ on a held-out set is

$$\text{PI}(j) = s(f, \mathcal{D}) - \mathbb{E}_{\pi}\left[s(f, \mathcal{D}^{\pi_j})\right]$$

where $\mathcal{D}^{\pi_j}$ denotes the dataset with column $j$ randomly permuted. Permuting a feature breaks its association with the target while leaving everything else intact; a large drop in score signals that the feature was important. This estimate is model-agnostic, unbiased with respect to cardinality, and works on any held-out set — though it is more expensive than impurity-based importance ($O(d)$ model evaluations) and can underestimate correlated features.

Comparing impurity-based and permutation importances on the breast cancer random forest:

In [ ]:
from sklearn.inspection import permutation_importance

# Impurity-based importances (from training)
imp_based = rf.feature_importances_
feat_names = data.feature_names

# Permutation importances on the test set
perm_result = permutation_importance(
    rf, X_test, y_test, n_repeats=20, random_state=42, n_jobs=-1
)
perm_imp = perm_result.importances_mean
perm_std = perm_result.importances_std

Side-by-side bar charts for the top 10 features by each method:

In [ ]:
#| code-fold: true
top_k = 10
idx_imp  = np.argsort(imp_based)[::-1][:top_k]
idx_perm = np.argsort(perm_imp)[::-1][:top_k]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.barh(range(top_k), imp_based[idx_imp][::-1], color="C0")
ax1.set_yticks(range(top_k))
ax1.set_yticklabels([feat_names[i] for i in idx_imp[::-1]], fontsize=8)
ax1.set_xlabel("mean decrease in impurity")
ax1.set_title("Impurity-based importance")
ax1.grid(axis="x", linestyle="dotted", alpha=0.6)

ax2.barh(range(top_k), perm_imp[idx_perm][::-1], xerr=perm_std[idx_perm][::-1],
         color="C1", capsize=3)
ax2.set_yticks(range(top_k))
ax2.set_yticklabels([feat_names[i] for i in idx_perm[::-1]], fontsize=8)
ax2.set_xlabel("mean decrease in accuracy")
ax2.set_title("Permutation importance (test set)")
ax2.grid(axis="x", linestyle="dotted", alpha=0.6)

fig.tight_layout();

**Figure.** The two rankings often agree on the top features but can diverge significantly for correlated or high-cardinality features. Error bars on the permutation chart reflect variability across the 20 permutation repeats.

## SHAP

Both impurity-based and permutation importance are **global** summaries: they collapse all samples into a single importance score per feature. They cannot answer the question: *why did the model predict this particular value for this particular sample?* **SHAP** (SHapley Additive exPlanations) provides both local and global answers grounded in cooperative game theory.

**Shapley values.** Treat each feature as a player and the model's prediction as the payout. The Shapley value of feature $j$ for input $\mathbf{x}$ is its fair contribution, averaged over all possible orderings in which features could be introduced:

$$\phi_j(\mathbf{x}) = \sum_{S \subseteq [d] \setminus \{j\}} \frac{|S|! \, (d - |S| - 1)!}{d!} \left[ v(S \cup \{j\}) - v(S) \right]$$

where $v(S)$ is the expected model output when only the features in $S$ are known (the remaining features are marginalized out). The Shapley values satisfy four axioms — **efficiency** ($\sum_j \phi_j = f(\mathbf{x}) - \mathbb{E}[f]$), **symmetry**, **dummy** (zero contribution for irrelevant features), and **linearity** — that together uniquely characterize them as the unique fair attribution satisfying these properties. The efficiency axiom is particularly useful: SHAP values sum exactly to the model's deviation from its baseline prediction, making them interpretable as additive contributions.

**TreeSHAP.** Naively computing Shapley values requires $2^d$ evaluations of $v(S)$. Lundberg et al. (2018) showed that for tree ensembles, the exact Shapley values can be computed in $O(T L d^2)$ time, where $T$ is the number of trees and $L$ is the maximum number of leaves, by exploiting the recursive structure of trees to propagate sample weights through the tree without enumerating all feature subsets. This makes SHAP tractable for large random forests and boosted ensembles.

**Global vs. local explanations.** A single SHAP value $\phi_j(\mathbf{x}^{(i)})$ is a **local explanation** for one sample. Aggregating $|\phi_j(\mathbf{x}^{(i)})|$ over all samples gives the **mean absolute SHAP** — a global feature importance that is unaffected by cardinality bias. The **beeswarm summary plot** combines both: each row is a feature ranked by mean $|\phi_j|$, each dot is one sample colored by raw feature value, and horizontal position encodes the SHAP value. This lets us see simultaneously which features matter globally and how they affect predictions (positively or negatively) as a function of their value.

We fit an `XGBRegressor` on California housing and compute exact SHAP values via `TreeExplainer`:

In [ ]:
import shap
import pandas as pd

# Use named features for readable SHAP plots
feature_names = housing.feature_names
X_shap_train = pd.DataFrame(X_h_train, columns=feature_names)
X_shap_test  = pd.DataFrame(X_h_test,  columns=feature_names)

# Refit on the full training split (no eval_set) for a clean explanation model
xgb_shap = xgb.XGBRegressor(
    n_estimators=xgb_model.best_iteration,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    verbosity=0,
)
xgb_shap.fit(X_shap_train, y_h_train)

explainer   = shap.TreeExplainer(xgb_shap)
shap_values = explainer.shap_values(X_shap_test)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Mean |SHAP| per feature (top 5):")
mean_abs = np.abs(shap_values).mean(axis=0)
ranked = sorted(zip(feature_names, mean_abs), key=lambda x: x[1], reverse=True)
for name, val in ranked[:5]:
    print(f"  {name:15s}: {val:.4f}")

The beeswarm summary plot for the top 10 features by mean $|\phi_j|$:

In [ ]:
#| label: fig-shap-summary
#| fig-cap: "SHAP beeswarm summary plot for an XGBoost model trained on California housing. Each row is a feature ranked by mean |SHAP|; each dot is a sample colored by feature value. The x-axis shows the SHAP value (impact on model output)."
#| code-fold: true

shap.summary_plot(
    shap_values,
    X_shap_test,
    plot_type="dot",
    max_display=10,
    show=False,
)
plt.tight_layout()
plt.show();

**Figure.** `MedInc` (median income) dominates — high values (pink) push predictions upward, low values (blue) push predictions down. `Latitude` and `Longitude` exhibit nonlinear patterns visible from the color gradient along the SHAP axis, reflecting the geographic variation in housing prices along the California coast.

:::{.callout-note}
The efficiency axiom guarantees that SHAP values are conservative: for each sample, $\sum_j \phi_j(\mathbf{x}) = f(\mathbf{x}) - \mathbb{E}[f].$ This means every prediction is fully decomposed into feature contributions with no unexplained residual — unlike permutation importance, which measures correlation with the output rather than additive attribution.

:::

For a local explanation of a single prediction, `shap.waterfall_plot` decomposes one sample's deviation from the baseline into individual feature contributions, showing exactly which features pushed the model up or down and by how much.

Local explanation for a single test sample:

In [ ]:
#| code-fold: true
# Build a shap.Explanation object for waterfall plotting
explanation = shap.Explanation(
    values=shap_values[0],
    base_values=explainer.expected_value,
    data=X_shap_test.iloc[0].values,
    feature_names=feature_names,
)
shap.waterfall_plot(explanation, show=False)
plt.tight_layout()
plt.show();

**Figure.** The waterfall chart traces how each feature pushes the prediction (red bars: positive contribution; blue bars: negative) away from the baseline $\mathbb{E}[f]$ toward the final prediction $f(\mathbf{x})$. The bars sum exactly to $f(\mathbf{x}) - \mathbb{E}[f]$ by the efficiency axiom.

---

■